# **P9.1 - Project 1: Customer Sentiment Analysis for Market Expansion**

## **Business Scenario**

My company is planning to expand into a new product category. As a data analyst, i've been hired to analyze customer reviews from Amazon to understand:

1. **Customer Pain Points**: What frustrates customers about products in this category?
2. **Positive Attributes**: What do customers love about these products?
3. **Temporal Trends**: How have sentiments and issues evolved over time?
4. **Product Quality Patterns**: What distinguishes highly-rated products from poorly-rated ones?
5. **Market Opportunities**: What gaps or opportunities exist in the market?

---



---

## **Deliverables**

1. **Jupyter Notebook** with complete analysis following the structure below
2. **Saved insights file** (JSON format) with extracted pain points and positives
3. **Visualizations** showing key findings
4. **Executive Summary** with business recommendations

---

---


---

## **Part 0: Understanding Parallel Processing**

### **Why Parallel Processing?**

When analyzing thousands of reviews, calling the LLM sequentially (one after another) would take **hours**. Parallel processing allows us to process multiple reviews **simultaneously**, dramatically reducing total time.

### **Sequential vs Parallel: A Simple Analogy**

Imagine you need to send 1000 letters:

**Sequential Processing (One at a time)**:
```
Letter 1 → Post Office → Wait for confirmation → Letter 2 → Post Office → Wait...
Time: 1000 letters × 2 minutes each = 2000 minutes (33+ hours)
```

**Parallel Processing (10 workers simultaneously)**:
```
Worker 1: Letters 1-100   ┐
Worker 2: Letters 101-200 │
Worker 3: Letters 201-300 ├─ All working at the same time!
...                       │
Worker 10: Letters 901-1000┘

Time: 100 letters × 2 minutes = 200 minutes (3.3 hours)
```

**Result**: 10x faster! 

---

### **How It Works in Python**

We use **asyncio** and **aiohttp** for asynchronous HTTP requests:

1. **Async Functions**: Functions that can "pause" while waiting for responses
2. **Concurrent Requests**: Send multiple API requests without waiting for each to complete
3. **Gather Results**: Collect all responses when they're ready

**Key Components**:
```python
async def process_review(review):      # 'async' keyword marks function as asynchronous
    response = await api_call(review)  # 'await' pauses THIS function (but others keep running)
    return response

# Process many reviews concurrently
tasks = [process_review(r) for r in reviews]  # Create tasks
results = await asyncio.gather(*tasks)          # Run all tasks in parallel
```

---

### **Important Considerations**

1. **API Rate Limits**: Don't send too many requests at once
   - Use `CONCURRENT_REQUESTS` to control batch size (typically 10-50)
   - Process in batches to avoid overwhelming the API

2. **Error Handling**: Some requests may fail
   - Always check if response is valid before processing
   - Consider retry logic for failed requests

3. **Progress Tracking**: Use `tqdm` to monitor progress
   - Shows how many batches completed
   - Estimates remaining time

4. **Saving Checkpoints**: For large datasets
   - Save results periodically (e.g., every 100 batches)
   - Allows recovery if process is interrupted

---

### **Example: Sequential vs Async**

**Sequential (SLOW)**:
```python
results = []
for review in reviews:  # Process one at a time
    result = process_review_sync(review)  # Wait for completion
    results.append(result)  # Then move to next
# Time: N reviews × avg_time per review
```

**Parallel (FAST)**:
```python
async def process_batch(reviews_batch):
    tasks = [process_review_async(r) for r in reviews_batch]  # Create all tasks
    return await asyncio.gather(*tasks)  # Run simultaneously!

# Process in batches of 10
for i in range(0, len(reviews), 10):
    batch = reviews[i:i+10]
    batch_results = await process_batch(batch)
    results.extend(batch_results)
# Time: (N reviews / batch_size) × avg_time per review
```

---

### **Real Performance Example**

Processing 5,000 reviews:
- Sequential: ~2 seconds/review = **10,000 seconds (2.8 hours)**
- Parallel (20 concurrent): ~2 seconds/batch of 20 = **500 seconds (8 minutes)**

**20x faster!** 

---

## **Part 1: Setup and Data Loading**

### **TODO 1.1**: Import required libraries



In [1]:
# Core Libraries
!pip install -q seaborn
!pip install -q plotly
!pip install -q tqdm
!pip install -q aiohttp
!pip install -q openai
!pip install -q FlagEmbedding
!pip install -q scikit-learn
!pip install -q matplotlib
!pip install -q python-dotenv
!pip install -q nbformat --upgrade
!pip install -q kaleido

import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from typing import List, Dict, Any
from tqdm.auto import tqdm
import asyncio
import aiohttp

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ML and Embeddings
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from collections import Counter
from FlagEmbedding import BGEM3FlagModel

# LLM API
import openai
from dotenv import load_dotenv

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Load environment variables (explicit project-local .env)
env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path)

print("✓ Libraries loaded successfully!")

/Users/ankitbhandari/miniforge3/envs/bsan885/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries loaded successfully!


### **TODO 1.2**: Configure constants

**Update the following**:
1. `DATA_FILE`: Change to your selected category file
2. `LLM_MODEL`: Choose an appropriate model from OpenRouter
3. `CONCURRENT_REQUESTS`: Start with 10, adjust based on API rate limits

In [ ]:


#
DATA_FILE = "amazon_reviews_us_Electronics_v1_00_filtered.tsv"

LLM_MODEL = "google/gemini-2.5-flash-lite-preview-09-2025"  # Example: fast and free
# Other options:
# - "openai/gpt-4o-mini" (balanced, paid)
# - "meta-llama/llama-3.2-3b-instruct:free" (free, smaller)

# Embedding model (pre-configured, no changes needed)
# Embedding model for clustering (Part 6)
# Option 1: Local model (requires GPU or significant CPU, no API cost)
#EMBEDDING_MODEL = "BAAI/bge-m3"
# Option 2: API-based model via OpenRouter (no GPU needed, small API cost)
EMBEDDING_MODEL = "openai/text-embedding-3-small"
# If using Option 2, see Module 8 notebooks for how to call the OpenRouter embeddings API.

# Parallel processing configuration
CONCURRENT_REQUESTS = 10  # Number of reviews to process simultaneously

print(f"✓ Configuration set:")
print(f"  - Data file: {DATA_FILE}")
print(f"  - LLM model: {LLM_MODEL}")
print(f"  - Concurrent requests: {CONCURRENT_REQUESTS}")

✓ Configuration set:
  - Data file: amazon_reviews_us_Electronics_v1_00_filtered.tsv
  - LLM model: google/gemini-2.5-flash-lite-preview-09-2025
  - Concurrent requests: 10


### **HELPER FUNCTION**: Load data



**What it does**:
1. Loads TSV file with proper error handling
2. Converts review_date to datetime
3. Extracts year and year_month for temporal analysis
4. Displays basic statistics

In [6]:
def load_review_data(filepath):
    """
    Load and preprocess Amazon review data from TSV file.
    
    Args:
        filepath: Path to the TSV file
        
    Returns:
        pandas DataFrame with loaded and preprocessed data
    """
    print(f"Loading data from: {filepath}")
    
    # Load TSV file
    df = pd.read_csv(filepath, sep='\t', on_bad_lines='skip', low_memory=False)
    
    # Convert review_date to datetime
    df['review_date'] = pd.to_datetime(df['review_date'], errors='coerce')
    df['year'] = df['review_date'].dt.year
    df['year_month'] = df['review_date'].dt.to_period('M')
    
    # Display overview
    print(f"\n✓ Dataset Overview:")
    print(f"  - Total reviews: {len(df):,}")
    print(f"  - Date range: {df['review_date'].min()} to {df['review_date'].max()}")
    print(f"  - Star ratings: {df['star_rating'].min()} to {df['star_rating'].max()}")
    print(f"  - Columns: {', '.join(df.columns.tolist())}")
    
    return df

# Load your data
df = load_review_data(DATA_FILE)

# Display sample
print("\nSample reviews:")
df.head()

Loading data from: amazon_reviews_us_Electronics_v1_00_filtered.tsv

✓ Dataset Overview:
  - Total reviews: 99,996
  - Date range: 2008-01-01 00:00:00 to 2015-08-31 00:00:00
  - Star ratings: 1 to 5
  - Columns: marketplace, customer_id, review_id, product_id, product_parent, product_title, product_category, star_rating, helpful_votes, total_votes, vine, verified_purchase, review_headline, review_body, review_date, year, year_month

Sample reviews:


,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date,year,year_month
0,US,20693829,R1EUO62UQI0UL7,B000II6YEA,497873594,Logitech Harmony 1000 Advanced Universal Remot...,Electronics,2,2,2,N,Y,Very Disappointing...,I purchased this remote at an EXCELLENT price ...,2008-01-20,2008,2008-01
1,US,25551684,R2EE4E719Y2AOW,B0015HS1HQ,799442557,Sony HTCT100 Sound Bar with Subwoofer - Black ...,Electronics,4,1,2,N,N,Good sound for its price.,This is a great product for audiophiles on a b...,2008-09-18,2008,2008-09
2,US,51909220,R1IO8IMWEM4QBN,B000I2QJB4,139059552,8-OUTLET Home Theater Surge Suppressor with Sm...,Electronics,1,2,3,N,Y,"Low Quality, Feels cheap; Too many lights",I purchased this power station to better organ...,2008-05-06,2008,2008-05
3,US,31617483,R3C0AUKSYMQPMU,B000FOYMKU,820064716,Coby CVE92 Isolation Stereo Earphones,Electronics,2,0,0,N,Y,Nothing special,From all the reviews I was excited and expecte...,2008-08-04,2008,2008-08
4,US,52080678,RFVANXHGU6KMD,B000MSQ8YM,924858386,Sony CMTBX5BT CMT Micro Component System with ...,Electronics,4,3,3,N,Y,Decent deal,This is not a high end microcomponent but some...,2008-03-03,2008,2008-03


---

## **Part 2: Initialize LLM and Embedding Models**


In [7]:
def setup_llm_client():
    """
    Initialize OpenRouter API client.
    Requires OPENROUTER_API_KEY environment variable.
    """
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise ValueError(
            "OPENROUTER_API_KEY not found in environment variables.\n"
            "Please create a .env file with: OPENROUTER_API_KEY=your_key_here"
        )
    return openai.OpenAI(base_url='https://openrouter.ai/api/v1', api_key=api_key)

# Initialize LLM client
llm_client = setup_llm_client()
print("✓ LLM client initialized!")

✓ LLM client initialized!


In [8]:
def initialize_embedder(model_name=EMBEDDING_MODEL):
    """
    Initialize the embedding model (BGE-M3).
    Uses GPU if available, otherwise CPU.
    
    Reference: M7.1 (embedding initialization)
    """
    import torch
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    embedder = BGEM3FlagModel(model_name, use_fp16=(device == "cuda"))
    print(f"✓ Embedding model '{model_name}' initialized!")
    
    return embedder

# Initialize embedding model
#embedder = initialize_embedder()


# --- Alternative: API-based embedding via OpenRouter ---
def embed_texts_api(texts, model="openai/text-embedding-3-small"):
    """Generate embeddings using OpenRouter API (same approach as Module 8)."""
    import openai
    client = openai.OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ.get("OPENROUTER_API_KEY"),
    )
    all_embeddings = []
    for i in range(0, len(texts), 100):
        batch = texts[i:i+100]
        response = client.embeddings.create(model=model, input=batch)
        all_embeddings.extend([item.embedding for item in response.data])
    return all_embeddings


### **HELPER FUNCTIONS**: LLM message sending (sync and async)



**Two versions**:
1. `send_message()`: Synchronous (blocking) - use for single calls or testing
2. `send_message_async()`: Asynchronous (non-blocking) - use for parallel processing



In [10]:
def send_message(messages, model=LLM_MODEL, max_tokens=2000, temperature=0.1, response_format=None):
    """
    Send a synchronous message to the LLM.
    Use this for single API calls or testing.
    
    Args:
        messages: List of message dicts with 'role' and 'content'
        model: Model identifier
        max_tokens: Maximum response length
        temperature: Creativity (0.0-1.0, lower = more deterministic)
        response_format: Optional dict specifying format (e.g., {"type": "json_object"})
        
    Returns:
        String response or None if error
    """
    try:
        completion_params = {
            "model": model,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": temperature
        }
        if response_format:
            completion_params["response_format"] = response_format

        response = llm_client.chat.completions.create(**completion_params)
        
        if response and response.choices and len(response.choices) > 0:
            if response.choices[0].message and response.choices[0].message.content:
                return response.choices[0].message.content.strip()
        return None
    except Exception as e:
        print(f"Error calling LLM: {e}")
        return None


async def send_message_async(session, messages, model=LLM_MODEL, max_tokens=2000, temperature=0.1, response_format=None):
    """
    Send an asynchronous message to the LLM.
    Use this for parallel processing of multiple reviews.
    
    Args:
        session: aiohttp.ClientSession instance
        messages: List of message dicts
        model: Model identifier
        max_tokens: Maximum response length
        temperature: Creativity setting
        response_format: Optional format specification
        
    Returns:
        String response or None if error
    """
    api_key = os.environ.get("OPENROUTER_API_KEY")
    url = "https://openrouter.ai/api/v1/chat/completions"
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature
    }
    
    if response_format:
        payload["response_format"] = response_format
    
    try:
        async with session.post(url, json=payload, headers=headers) as response:
            if response.status == 200:
                data = await response.json()
                if data.get("choices") and len(data["choices"]) > 0:
                    return data["choices"][0]["message"]["content"].strip()
            else:
                error_text = await response.text()
                print(f"API Error {response.status}: {error_text}")
                return None
    except Exception as e:
        print(f"Error in async LLM call: {e}")
        return None

print("✓ Helper functions defined (sync and async)!")

✓ Helper functions defined (sync and async)!


---

## **Part 3: Exploratory Data Analysis**

### **TODO 3.1**: Create visualizations

**Your task**: Create the following visualizations to understand your data:

1. **Rating Distribution**: Bar chart showing how many reviews for each star rating
2. **Ratings Over Time**: Line chart showing average rating by month/year
3. **Review Volume Over Time**: Line chart showing number of reviews by month/year

**Hints**:
- Use `plotly.graph_objects` or `plotly.express` for interactive plots
- Reference: M9.1 Example (Part 3)
- For temporal aggregation, use `df.groupby('year_month')` or `df.groupby('year')`

In [11]:
# ── 1. Rating Distribution ──────────────────────────────────────────
# WHY: Shows whether the dataset is skewed (spoiler: most Amazon reviews
#       are 5-star, so "positive" noise is high → pain-point extraction
#       from the 1-2 star slice will be especially valuable later).

rating_counts = df['star_rating'].value_counts().sort_index()

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        'Rating Distribution',
        'Average Rating Over Time',
        'Review Volume Over Time'
    )
)

fig.add_trace(
    go.Bar(
        x=rating_counts.index.astype(str),
        y=rating_counts.values,
        marker_color=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60'],
        text=rating_counts.values,
        textposition='outside',
        name='Rating Count'
    ),
    row=1, col=1
)

# ── 2. Average Rating Over Time ────────────────────────────────────
# WHY: If average rating is falling, the category may have a quality
#       problem → bigger market opportunity for a new entrant.

monthly_stats = (
    df.groupby('year')
    .agg(avg_rating=('star_rating', 'mean'), review_count=('review_id', 'count'))
    .reset_index()
)

fig.add_trace(
    go.Scatter(
        x=monthly_stats['year'],
        y=monthly_stats['avg_rating'],
        mode='lines+markers',
        name='Avg Rating',
        line=dict(color='#3498db', width=2)
    ),
    row=1, col=2
)

# ── 3. Review Volume Over Time ─────────────────────────────────────
# WHY: Growing volume = growing market. Declining = saturated/dying.

fig.add_trace(
    go.Bar(
        x=monthly_stats['year'],
        y=monthly_stats['review_count'],
        name='Review Count',
        marker_color='#9b59b6'
    ),
    row=1, col=3
)

fig.update_layout(
    height=450, width=1200,
    title_text='Electronics Reviews — Exploratory Overview',
    showlegend=False
)
fig.update_xaxes(title_text='Star Rating', row=1, col=1)
fig.update_xaxes(title_text='Year', row=1, col=2)
fig.update_xaxes(title_text='Year', row=1, col=3)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Avg Rating', row=1, col=2)
fig.update_yaxes(title_text='# Reviews', row=1, col=3)

fig.show()

# Quick summary stats
print(f"\nDataset Summary:")
print(f"  Total reviews: {len(df):,}")
print(f"  Average rating: {df['star_rating'].mean():.2f}")
print(f"  Median rating:  {df['star_rating'].median():.0f}")
print(f"  Unique products: {df['product_title'].nunique():,}")
print(f"  Date range: {df['review_date'].min().date()} → {df['review_date'].max().date()}")

# Brief business interpretation for the opening EDA
high_share = (df['star_rating'] >= 4).mean() * 100
low_share = (df['star_rating'] <= 2).mean() * 100
print("\nInterpretation:")
print(f"  - The category is positivity-skewed ({high_share:.1f}% are 4-5 stars), so top-line sentiment appears strong.")
print(f"  - Low ratings are still material ({low_share:.1f}% are 1-2 stars), making pain-point mining strategically useful.")
print("  - Yearly volume/rating trends indicate whether demand is growing and where a new entrant can differentiate.")



Dataset Summary:
  Total reviews: 99,996
  Average rating: 4.00
  Median rating:  5
  Unique products: 28,490
  Date range: 2008-01-01 → 2015-08-31

Interpretation:
  - The category is positivity-skewed (74.0% are 4-5 stars), so top-line sentiment appears strong.
  - Low ratings are still material (18.1% are 1-2 stars), making pain-point mining strategically useful.
  - Yearly volume/rating trends indicate whether demand is growing and where a new entrant can differentiate.


### **TODO 3.2**: Identify top and bottom products

**Your task**: 
1. Group reviews by `product_title`
2. Calculate average rating and review count for each product
3. Filter to products with at least 10 reviews
4. Identify top 10 highest-rated and bottom 10 lowest-rated products

**Hints**:
- Use `df.groupby('product_title').agg({...})`
- Use `.nlargest()` and `.nsmallest()`


In [12]:
# WHY: We need these lists for Part 8 (LLM comparison of winners vs losers).
#       Filtering to >=10 reviews avoids products with one fluke review.

MIN_REVIEWS = 10

product_stats = (
    df.groupby('product_title')
    .agg(
        avg_rating=('star_rating', 'mean'),
        review_count=('star_rating', 'count'),
        helpful_votes=('helpful_votes', 'sum')
    )
    .reset_index()
)
product_stats = product_stats[product_stats['review_count'] >= MIN_REVIEWS]

top_products = product_stats.nlargest(10, 'avg_rating')
bottom_products = product_stats.nsmallest(10, 'avg_rating')

# Store the product title lists for Part 8
top_product_titles = top_products['product_title'].tolist()
bottom_product_titles = bottom_products['product_title'].tolist()

print("═" * 80)
print("TOP 10 HIGHEST-RATED PRODUCTS (≥10 reviews)")
print("═" * 80)
for i, (_, row) in enumerate(top_products.iterrows(), 1):
    print(f"  {i:2d}. ★ {row['avg_rating']:.2f}  ({row['review_count']:3d} reviews)  {row['product_title'][:75]}")

print(f"\n{'═' * 80}")
print("BOTTOM 10 LOWEST-RATED PRODUCTS (≥10 reviews)")
print("═" * 80)
for i, (_, row) in enumerate(bottom_products.iterrows(), 1):
    print(f"  {i:2d}. ★ {row['avg_rating']:.2f}  ({row['review_count']:3d} reviews)  {row['product_title'][:75]}")

════════════════════════════════════════════════════════════════════════════════
TOP 10 HIGHEST-RATED PRODUCTS (≥10 reviews)
════════════════════════════════════════════════════════════════════════════════
   1. ★ 5.00  ( 14 reviews)  BlueRigger 3.5mm Male to Female Stereo Audio Cable (6 Feet)
   2. ★ 5.00  ( 23 reviews)  BlueRigger Digital Optical Audio Toslink Cable(10 feet)
   3. ★ 5.00  ( 10 reviews)  Bluecell Black/Blue/Pink Earphone in-ear Hard Case/Bag Pack of 3
   4. ★ 5.00  ( 10 reviews)  C2G/Cables to Go 98038 Premium Toslink Optical Digital Audio Cable (8 Feet,
   5. ★ 5.00  ( 11 reviews)  Caseling Premium Hard EVA Case Travel Bag Pouch for Bose Soundlink Mini Blu
   6. ★ 5.00  ( 29 reviews)  Kingston Apple 667MHz DDR2 SoDimm iMac and Macbook Memory
   7. ★ 5.00  ( 10 reviews)  OSD Audio OSD-HDMI-Value-V14 High Speed HDMI Cable with Ethernet V1.4
   8. ★ 5.00  ( 15 reviews)  OmniMount UCLX-P Articulating Wall Mount for 42" to 63" Displays (Silver)
   9. ★ 4.96  ( 26 reviews)

---

## **Part 4: Structured LLM Analysis**

### **Understanding the Extraction Process**

We'll use the LLM to extract structured insights from each review:

**Input**: Raw review text
```
"This game is amazing! Graphics are stunning and gameplay is smooth.
Only issue is occasional lag during multiplayer."
```

**Output**: Structured JSON
```json
{
  "pain_points": ["multiplayer lag"],
  "positive_aspects": ["stunning graphics", "smooth gameplay"],
  "main_themes": ["game quality", "performance"],
  "sentiment": "positive"
}
```

**Why 2-3 words maximum?**
- Enables aggregation across reviews
- "download was very slow" → "slow download"
- "took forever to install" → "slow installation"
- Both can be grouped as similar issues

---

### **HELPER FUNCTION**: Extract review insights (async)

**This function is provided for you** - but you should understand how it works.

**Key components**:
1. **Prompt engineering**: Clear instructions for structured output
2. **JSON mode**: Ensures LLM returns valid JSON
3. **Error handling**: Gracefully handles invalid responses
4. **Metadata**: Attaches review context to results

In [16]:
import numbers

def _make_json_safe(obj):
    """Recursively convert numpy/pandas scalars so json.dumps / json.dump never hits int64/float64."""
    if isinstance(obj, dict):
        return {k: _make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_make_json_safe(x) for x in obj]
    if obj is None or isinstance(obj, bool):
        return obj
    if isinstance(obj, str):
        return obj
    if isinstance(obj, numbers.Integral):
        return int(obj)
    if isinstance(obj, numbers.Real):
        return float(obj)
    if hasattr(obj, "item"):
        try:
            return _make_json_safe(obj.item())
        except Exception:
            pass
    return obj


async def extract_review_insights_async(session, review_text, rating, product_title, review_id, year):
    """
    Extract structured insights from a review using LLM (async version for parallel processing).
    
    Args:
        session: aiohttp.ClientSession
        review_text: The review text to analyze
        rating: Star rating (1-5)
        product_title: Product name
        review_id: Unique review identifier
        year: Review year for temporal analysis
        
    Returns:
        Dict with extracted insights or None if error
    """
    prompt = f"""You are analyzing a customer review for a product.

Product: {product_title}
Rating: {rating}/5 stars
Review: {review_text}

Extract the following information in JSON format. IMPORTANT: Use ONLY 2-3 words maximum for each item in the lists to enable aggregation across reviews.

{{
  "pain_points": ["2-3 word descriptions of problems, e.g. 'download issues', 'poor quality', 'breaks easily'"],
  "positive_aspects": ["2-3 word descriptions of positives, e.g. 'great quality', 'fast shipping', 'good value'"],
  "main_themes": ["2-3 word topics, e.g. 'product quality', 'customer service', 'durability'"],
  "sentiment": "positive/negative/mixed/neutral",
  "purchase_decision_factors": ["2-3 word factors, e.g. 'price point', 'brand reputation', 'features'"]
}}

CRITICAL RULES:
- Each item in lists must be 2-3 words MAXIMUM (not full sentences)
- Use consistent, generic terminology
- Be specific but concise
- Use lowercase
- If nothing is mentioned for a category, use an empty list
"""

    messages = [{"role": "user", "content": prompt}]
    
    response = await send_message_async(
        session,
        messages,
        max_tokens=800,
        temperature=0.1,
        response_format={"type": "json_object"}
    )
    
    if response:
        try:
            insights = json.loads(response)
            # Attach metadata (native int/str — pandas columns are numpy int64 and break json.dumps)
            insights['review_id'] = str(review_id) if pd.notna(review_id) else None
            insights['star_rating'] = int(rating) if pd.notna(rating) else None
            insights['year'] = int(year) if pd.notna(year) else None
            insights['product_title'] = product_title
            insights['review_body'] = review_text
            return _make_json_safe(insights)
        except json.JSONDecodeError:
            return None
    return None

print("✓ Extraction function defined (async)!")

✓ Extraction function defined (async)!


### **TODO 4.1**: Test the extraction function

**Your task**: Test the extraction function on a single review before processing all reviews.

**Why test first?**
- Verify the prompt works correctly
- Check that JSON parsing succeeds
- Ensure output quality is acceptable
- Saves time and API costs before full processing

**Hints**:
- Select one review: `test_review = df.iloc[0]`
- For async function, you need to use `await` inside an async context
- Use `asyncio.run()` to run async code in Jupyter


In [17]:
# WHY TEST FIRST? One LLM call costs fractions of a cent and takes ~2 seconds.
# Running 10K calls costs real money and takes ~30 min. Always verify the
# prompt works and returns clean JSON before scaling up.

# Pick a review with enough text to produce meaningful insights
# .fillna('') handles the 2 NaN rows in review_body
test_review = df[df['review_body'].fillna('').str.len() > 100].iloc[0]

print("Test Review:")
print(f"  Product: {test_review['product_title']}")
print(f"  Rating:  {test_review['star_rating']}/5")
print(f"  Text:    {str(test_review['review_body'])[:500]}")
print("\n" + "=" * 80 + "\n")

async def test_extraction():
    async with aiohttp.ClientSession() as session:
        result = await extract_review_insights_async(
            session,
            str(test_review['review_body']),
            test_review['star_rating'],
            str(test_review['product_title']),
            test_review['review_id'],
            test_review['year']
        )
        return result

test_result = await test_extraction()

if test_result:
    print("Extracted Insights:")
    print(json.dumps(test_result, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
    print("\n✓ Extraction works! JSON parsed successfully.")
else:
    print("✗ Extraction returned None — check your API key and model.")

Test Review:
  Product: Logitech Harmony 1000 Advanced Universal Remote (Silver) (Discontinued by Manufacturer)
  Rating:  2/5
  Text:    I purchased this remote at an EXCELLENT price from Amazon, as usual (GRRR8 Prices and Service), to interoperate with my Samsung LNT4671F, Samsung DVD and Dish HD receiver. Anyway, I have to report that to date this has been my \\"MOST\\" disappointing purchase ever on the web or else wise. The remote was super easy to configure but from day one I noticed that the software driven remote was not as intuitive as I'd hoped for and in 2 - 3 weeks the remote just plain didn't work; it froze up and one


Extracted Insights:
{
  "pain_points": [
    "software not intuitive",
    "remote froze up",
    "stopped working"
  ],
  "positive_aspects": [
    "excellent price",
    "easy configure",
    "easy return"
  ],
  "main_themes": [
    "software usability",
    "product failure",
    "return process"
  ],
  "sentiment": "negative",
  "purchase_decision_fact

### **Understanding Parallel Processing Implementation**

Now we'll process ALL reviews using parallel processing. Here's how it works:

**Step 1: Prepare Reviews**
```python
reviews_to_process = []
for review in df:
    if review_is_valid(review):
        reviews_to_process.append(review_data)
```

**Step 2: Process in Batches**
```python
batch_size = 10  # Process 10 at once
for i in range(0, len(reviews), batch_size):
    batch = reviews[i:i+batch_size]
    results = await process_batch(batch)  # All 10 run in parallel!
```

**Step 3: Handle Errors & Save Checkpoints**
```python
if batch_num % 100 == 0:  # Every 100 batches
    save_checkpoint(results)  # Save progress
```

---

### **HELPER FUNCTION**: Parallel review processing

**This function is provided for you** - it handles the complex async orchestration.

In [18]:
async def process_reviews_batch(reviews_batch):
    """
    Process a batch of reviews concurrently.
    
    Args:
        reviews_batch: List of review dicts to process
        
    Returns:
        List of extracted insights (or None for failed extractions)
    """
    async with aiohttp.ClientSession() as session:
        tasks = []
        for review in reviews_batch:
            task = extract_review_insights_async(
                session,
                review['review_body'],
                review['star_rating'],
                review['product_title'],
                review['review_id'],
                review['year']
            )
            tasks.append(task)
        
        # Run all tasks in parallel and collect results
        results = await asyncio.gather(*tasks)
        return [r for r in results if r is not None]


async def process_all_reviews(reviews_list, batch_size=CONCURRENT_REQUESTS, checkpoint_interval=100):
    """
    Process all reviews in batches with progress tracking and checkpoints.
    
    Args:
        reviews_list: List of all reviews to process
        batch_size: Number of reviews to process concurrently
        checkpoint_interval: Save checkpoint every N batches
        
    Returns:
        List of all extracted insights
    """
    all_insights = []
    
    # Process in batches
    for i in tqdm(range(0, len(reviews_list), batch_size), desc="Processing batches"):
        batch = reviews_list[i:i + batch_size]
        batch_results = await process_reviews_batch(batch)
        all_insights.extend(batch_results)
        
        # Save checkpoint periodically
        if (i // batch_size + 1) % checkpoint_interval == 0:
            temp_df = pd.DataFrame(all_insights)
            temp_df.to_json('review_insights_checkpoint.json', orient='records', indent=2)
            print(f"\n✓ Checkpoint: Saved {len(all_insights)} insights...")
    
    return all_insights

print("✓ Parallel processing functions defined!")

✓ Parallel processing functions defined!


### **TODO 4.2**: Process ALL reviews

**Your task**: Run the parallel processing pipeline on all reviews.

**Before running**:
1. Estimate API costs: ~0.0005 cents per review depending on model
2. Estimate time: ~5-10 minutes for 5000 reviews with batch_size=10
3. Ensure you have disk space for checkpoints

**For testing**: Start with a small subset (e.g., first 100 reviews) to verify everything works.

**Hints**:
- Filter reviews by minimum length (e.g., >20 characters)
- Limit review text length to save tokens (e.g., first 2000 characters)
- Monitor the progress bar
- Check checkpoints are being saved


In [21]:
# ── Step 1: Build the list of reviews to send to the LLM ────────────
# WHY filter?  Very short reviews ("Great!", "Meh") give the LLM nothing
#   to extract.  Truncating to 2000 chars keeps token cost down while
#   preserving the meat of long reviews.
# WHY sample?  100K reviews × ~$0.0005 each ≈ $50.  A 5000-review sample
#   is statistically representative and costs ~$2.50.  You can increase
#   later if you want.

print("Preparing reviews for analysis...")

SAMPLE_SIZE = 30000 # Increase to None or len(df) for full dataset

reviews_to_process = []
for idx, row in df.iterrows():
    review_body = str(row['review_body'])[:2000]
    if len(review_body.strip()) >= 20:
        reviews_to_process.append({
            'review_body': review_body,
            'star_rating': row['star_rating'],
            'product_title': str(row['product_title']),
            'review_id': row['review_id'],
            'year': row['year']
        })

print(f"Reviews with ≥20 chars: {len(reviews_to_process):,}")

if SAMPLE_SIZE and SAMPLE_SIZE < len(reviews_to_process):
    import random
    random.seed(42)
    reviews_to_process = random.sample(reviews_to_process, SAMPLE_SIZE)
    print(f"Sampled down to: {len(reviews_to_process):,} reviews")

print(f"Estimated API calls: {len(reviews_to_process):,}")
print(f"Estimated time (batch_size={CONCURRENT_REQUESTS}): "
      f"~{len(reviews_to_process) * 2 / CONCURRENT_REQUESTS / 60:.1f} minutes")

# ── Step 2: Run parallel processing ────────────────────────────────
insights_list = await process_all_reviews(reviews_to_process)

print(f"\n✓ Extracted insights from {len(insights_list):,} reviews")
print(f"Success rate: {len(insights_list)/len(reviews_to_process)*100:.1f}%")

Preparing reviews for analysis...
Reviews with ≥20 chars: 96,515
Sampled down to: 30,000 reviews
Estimated API calls: 30,000
Estimated time (batch_size=10): ~100.0 minutes


Processing batches:   2%|▏         | 64/3000 [06:28<75:02:45, 92.02s/it]

Error in async LLM call: 


Processing batches:   3%|▎         | 100/3000 [07:16<57:24,  1.19s/it]  


✓ Checkpoint: Saved 999 insights...


Processing batches:   7%|▋         | 200/3000 [09:13<48:53,  1.05s/it]  

... (57 outputs left out)

Processing batches:  97%|█████████▋| 2900/3000 [1:01:45<01:53,  1.14s/it]


✓ Checkpoint: Saved 28942 insights...


Processing batches: 100%|██████████| 3000/3000 [1:03:40<00:00,  1.27s/it]


✓ Checkpoint: Saved 29942 insights...

✓ Extracted insights from 29,942 reviews
Success rate: 99.8%


### **TODO 4.3**: Save and load insights

**Your task**: 
1. Save the extracted insights to a JSON file
2. Load insights from the saved file (for future use)
3. Convert to pandas DataFrame for analysis

**Why save?**
- Avoid re-running expensive LLM processing
- Share results with others
- Create backups before further processing

In [19]:
# WHY SAVE? The LLM calls took minutes and cost money.
# Saving means you never have to re-run them — just load from disk.

INSIGHTS_FILE = "electronics_review_insights.json"
CLUSTERED_INSIGHTS_FILE = "electronics_review_insights_with_clusters.json"
# True = skip Part 4.2; load df after imports + TSV load (Part 1) only.
LOAD_INSIGHTS_FROM_DISK = True
PREFER_CLUSTERED_FILE = True

if LOAD_INSIGHTS_FROM_DISK:
    file_to_load = INSIGHTS_FILE
    if PREFER_CLUSTERED_FILE and os.path.exists(CLUSTERED_INSIGHTS_FILE):
        file_to_load = CLUSTERED_INSIGHTS_FILE
    insights_df = pd.read_json(file_to_load, orient='records')
    print(f"✓ Loaded {len(insights_df):,} insights from {file_to_load}")
else:
    insights_df = pd.DataFrame(insights_list)
    insights_df.to_json(INSIGHTS_FILE, orient='records', indent=2)
    print(f"✓ Saved {len(insights_df):,} insights to {INSIGHTS_FILE}")

print(f"\nColumns: {insights_df.columns.tolist()}")
insights_df.head(3)

✓ Loaded 29,942 insights from electronics_review_insights_with_clusters.json

Columns: ['pain_points', 'positive_aspects', 'main_themes', 'sentiment', 'purchase_decision_factors', 'review_id', 'star_rating', 'year', 'product_title', 'review_body', 'positive_points', 'main_points', 'cluster', 'tsne_x', 'tsne_y']


,pain_points,positive_aspects,main_themes,sentiment,purchase_decision_factors,review_id,star_rating,year,product_title,review_body,positive_points,main_points,cluster,tsne_x,tsne_y
0,"[wrong item received, missing cable, incorrect...","[seller replied, easy refund]","[product mismatch, customer service, return pr...",mixed,"[cable length, connector type]",R3MLEKA67QVUVV,2,2014,"GoldX, 6'/18"" 7-in-1 Quickconnect Kit (Catalog...","I was looking for \\""USB Mini B male (mini dev...",None,None,1,-25.258608,1.448148
1,[shipping tracking issue],"[works well, very happy]","[product function, shipping status]",positive,[product performance],RIR2TZ9KIN959,4,2009,NEW AC Adapter / Power Supply for Dell INSPIRO...,This battery works well so far. Amazon doesn'...,None,None,11,-51.513340,65.772301
2,"[camera turns off, door misalignment, button i...","[looks nice, buttons functional, works properly]","[battery grip, power loss, product modification]",mixed,[great deal],R3K07S0084NDXY,3,2008,Satechi LCD Timer Vertical Battery Grip for Ca...,I was debating getting a standard canon batter...,None,None,1,-30.086147,71.732735


---

## **Part 5: Aggregate and Analyze Extracted Insights**

### **TODO 5.1**: Count and display top pain points and positives

**Your task**: 
1. Extract all pain points from all reviews into a single list
2. Extract all positive aspects into a single list
3. Use `Counter` to count frequencies
4. Display top 20 of each

**Hints**:
- Use `from collections import Counter`
- Iterate through `insights_df` rows
- Each row has `pain_points` and `positive_aspects` as lists
- Use `.extend()` to add lists together


In [20]:
# WHY FLATTEN + COUNT?
# Each review produced a list like ["poor battery", "breaks easily"].
# We combine ALL reviews' lists into one big list and count how often
# each phrase appears.  The top items = the category's defining issues.
#
# NOTE: "none mentioned" is tracked separately as a coverage signal,
# not as an actual pain-point category.

NO_PAIN_TOKENS = {"none mentioned", "none", "n/a", "na", "no issues mentioned"}


def clean_list_items(items, blocked=None):
    if not isinstance(items, list):
        return []
    blocked = blocked or set()
    out = []
    for x in items:
        if not isinstance(x, str):
            continue
        x_norm = x.strip().lower()
        if not x_norm or x_norm in blocked:
            continue
        out.append(x_norm)
    return out


all_pain_points = []
all_positives = []
all_themes = []
all_factors = []
sentiment_list = []

# Business metric: reviews with no explicit pain point (after cleaning)
insights_df['pain_points_clean'] = insights_df['pain_points'].apply(
    lambda v: clean_list_items(v, blocked=NO_PAIN_TOKENS)
)
insights_df['has_explicit_pain'] = insights_df['pain_points_clean'].apply(lambda v: len(v) > 0)

for _, row in insights_df.iterrows():
    all_pain_points.extend(row.get('pain_points_clean', []))
    all_positives.extend(clean_list_items(row.get('positive_aspects')))
    all_themes.extend(clean_list_items(row.get('main_themes')))
    all_factors.extend(clean_list_items(row.get('purchase_decision_factors')))
    if isinstance(row.get('sentiment'), str):
        sentiment_list.append(row['sentiment'].strip().lower())

pain_counts = Counter(all_pain_points)
positive_counts = Counter(all_positives)
theme_counts = Counter(all_themes)
factor_counts = Counter(all_factors)
sentiment_counts = Counter(sentiment_list)


def print_top(title, counter, n=20):
    print(f"\n{'═' * 60}")
    print(f"  {title}")
    print(f"{'═' * 60}")
    for item, count in counter.most_common(n):
        bar = '█' * min(count // 10, 40)
        print(f"  {item:35s} {count:5d}  {bar}")


print_top("TOP 20 PAIN POINTS (EXCLUDING 'NONE MENTIONED')", pain_counts)
print_top("TOP 20 POSITIVE ASPECTS", positive_counts)
print_top("TOP 20 THEMES", theme_counts)
print_top("TOP 20 PURCHASE DECISION FACTORS", factor_counts)

print(f"\n{'═' * 60}")
print("  NO-PAIN COVERAGE METRIC")
print(f"{'═' * 60}")
no_pain_rate = (1 - insights_df['has_explicit_pain'].mean()) * 100
print(f"  Reviews with no explicit pain point: {no_pain_rate:.1f}%")

if 'star_rating' in insights_df.columns:
    print("\n  By rating segment:")
    for label, mask in [
        ("Low (1-2★)", insights_df['star_rating'] <= 2),
        ("Mid (3★)", insights_df['star_rating'] == 3),
        ("High (4-5★)", insights_df['star_rating'] >= 4),
    ]:
        seg = insights_df[mask]
        if len(seg) == 0:
            continue
        seg_no_pain = (1 - seg['has_explicit_pain'].mean()) * 100
        print(f"    {label:12s} {seg_no_pain:6.1f}% no explicit pain")

print(f"\n{'═' * 60}")
print("  SENTIMENT DISTRIBUTION")
print(f"{'═' * 60}")
for sent, count in sentiment_counts.most_common():
    pct = count / len(sentiment_list) * 100
    print(f"  {sent:15s} {count:5d}  ({pct:.1f}%)")



════════════════════════════════════════════════════════════
  TOP 20 PAIN POINTS (EXCLUDING 'NONE MENTIONED')
════════════════════════════════════════════════════════════
  stopped working                       405  ████████████████████████████████████████
  short lifespan                        399  ███████████████████████████████████████
  high price                            187  ██████████████████
  low volume                            186  ██████████████████
  did not work                          182  ██████████████████
  poor fit                              164  ████████████████
  poor reception                        143  ██████████████
  sound quality                         133  █████████████
  uncomfortable fit                     132  █████████████
  poor sound                            111  ███████████
  does not work                         110  ███████████
  short battery life                    110  ███████████
  poor quality                           94  ████████

### **TODO 5.2**: Visualize top pain points and positives

**Your task**: Create horizontal bar charts showing top categories for all dimensions you are identifying.

**Hints**:
- Use `matplotlib.pyplot` or `plotly`
- Create 2x2 subplot grid or similar, depending on number of dimensions
- Use horizontal bars for readability
- Sort bars by frequency

In [21]:
# WHY HORIZONTAL BARS? Long text labels read naturally on the y-axis.
# A 2×2 grid gives a single "dashboard" view of all four dimensions.

TOP_N = 15

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        f'Top {TOP_N} Pain Points',
        f'Top {TOP_N} Positive Aspects',
        f'Top {TOP_N} Themes',
        f'Top {TOP_N} Purchase Decision Factors'
    ),
    horizontal_spacing=0.25
)

def add_bar(counter, row, col, color):
    items = counter.most_common(TOP_N)[::-1]
    fig.add_trace(
        go.Bar(
            y=[i[0] for i in items],
            x=[i[1] for i in items],
            orientation='h',
            marker_color=color
        ),
        row=row, col=col
    )

add_bar(pain_counts,     1, 1, '#e74c3c')
add_bar(positive_counts, 1, 2, '#27ae60')
add_bar(theme_counts,    2, 1, '#3498db')
add_bar(factor_counts,   2, 2, '#9b59b6')

fig.update_layout(
    height=900, width=1200,
    title_text='Aggregated Review Insights — Electronics Category',
    showlegend=False
)
fig.show()

### **TODO 5.3**: Analyze by rating category

**Your task**: Compare pain points and positives across rating groups:
- Low-rated (1-2 stars)
- Mid-rated (3 stars)
- High-rated (4-5 stars)

**Questions to answer**:
- What are the top pain points for low-rated products?
- Do high-rated products have any pain points?
- What positive aspects are most common in high-rated products?

**Hints**:
- Filter insights_df by star_rating
- Create separate Counter objects for each group
- Compare top items across groups


In [22]:
# WHY SPLIT BY RATING?
# Low-rated (1-2★) reviews reveal what makes customers ANGRY.
# High-rated (4-5★) reviews reveal what makes them HAPPY.
# Mid-rated (3★) often contain the most nuanced "it's okay but..." feedback.
# Comparing these segments tells you: what to build and what to avoid.

low_rated  = insights_df[insights_df['star_rating'] <= 2]
mid_rated  = insights_df[insights_df['star_rating'] == 3]
high_rated = insights_df[insights_df['star_rating'] >= 4]

print(f"Low-rated (1-2★):  {len(low_rated):,} reviews")
print(f"Mid-rated (3★):    {len(mid_rated):,} reviews")
print(f"High-rated (4-5★): {len(high_rated):,} reviews")

def get_top_items(df_subset, column_name, top_n=10):
    items = []
    for _, row in df_subset.iterrows():
        if isinstance(row.get(column_name), list):
            items.extend(row[column_name])
    return Counter(items).most_common(top_n)

segments = {
    'Low-Rated (1-2★)': low_rated,
    'Mid-Rated (3★)': mid_rated,
    'High-Rated (4-5★)': high_rated
}

for label, segment in segments.items():
    print(f"\n{'═' * 60}")
    print(f"  {label}")
    print(f"{'═' * 60}")

    top_pains = get_top_items(segment, 'pain_points')
    top_pos   = get_top_items(segment, 'positive_aspects')

    print(f"\n  Pain Points:")
    for item, count in top_pains:
        print(f"    {item:30s} {count:4d}")

    print(f"\n  Positive Aspects:")
    for item, count in top_pos:
        print(f"    {item:30s} {count:4d}")

# Visualization: side-by-side comparison of pain points across segments
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Low (1-2★) Pain Points', 'Mid (3★) Pain Points', 'High (4-5★) Pain Points']
)

colors = ['#e74c3c', '#f39c12', '#27ae60']
for i, (label, segment) in enumerate(segments.items(), 1):
    items = get_top_items(segment, 'pain_points', 10)[::-1]
    fig.add_trace(
        go.Bar(
            y=[it[0] for it in items],
            x=[it[1] for it in items],
            orientation='h',
            marker_color=colors[i-1]
        ),
        row=1, col=i
    )

fig.update_layout(height=500, width=1400, showlegend=False,
                  title_text='Pain Points by Rating Segment')
fig.show()

Low-rated (1-2★):  5,621 reviews
Mid-rated (3★):    2,391 reviews
High-rated (4-5★): 21,930 reviews

════════════════════════════════════════════════════════════
  Low-Rated (1-2★)
════════════════════════════════════════════════════════════

  Pain Points:
    stopped working                 344
    short lifespan                  297
    did not work                    163
    does not work                   109
    poor fit                         95
    poor reception                   94
    poor sound                       81
    low volume                       79
    poor quality                     78
    broke quickly                    75

  Positive Aspects:
    none mentioned                  123
    low price                        71
    good sound                       59
    worked great                     56
    great sound                      50
    good price                       35
    worked initially                 31
    easy setup                       31
 

### **Unified Categorization — Pain Points AND Positive Features**

**The problem**: Hundreds of distinct 2-3 word phrases are hard to act on. We need higher-level **categories** that a product team can own.

**What we'll build**: A single unified category system that covers:
- **Pain Point categories** (what to fix) — e.g., "Battery & Power", "Build Quality"
- **Positive Feature categories** (what to double down on) — e.g., "Audio Excellence", "Value & Pricing"
- **"Others"** — anything that doesn't fit neatly

**Why unified?** Because seeing pain points and positives in the same framework reveals interesting patterns:
- A category that's both a top pain point AND a top positive (e.g., "Sound Quality") means **performance varies wildly across products** → opportunity to win by being consistent.
- A category that's all positives with zero pain points → customers take this for granted, don't mess it up.
- A category that's all pain points → the market hasn't solved this yet → biggest opportunity.

In [23]:
# ═══════════════════════════════════════════════════════════════════
# STEP 1: Send BOTH pain points and positives to LLM to create a
#          unified category system in one shot.
# ═══════════════════════════════════════════════════════════════════

# Safety fallback: if counts are missing (e.g., after kernel restart), rebuild from insights_df
if 'pain_counts' not in globals() or 'positive_counts' not in globals():
    print("pain_counts/positive_counts not found. Rebuilding from insights_df...")
    if 'insights_df' not in globals():
        raise NameError("insights_df is not defined. Run cell 26 first.")

    NO_PAIN_TOKENS = {"none mentioned", "none", "n/a", "na", "no issues mentioned"}

    def _clean_list_items(items, blocked=None):
        if not isinstance(items, list):
            return []
        blocked = blocked or set()
        out = []
        for x in items:
            if not isinstance(x, str):
                continue
            x_norm = x.strip().lower()
            if not x_norm or x_norm in blocked:
                continue
            out.append(x_norm)
        return out

    _all_pain = []
    _all_pos = []
    for _, _row in insights_df.iterrows():
        _all_pain.extend(_clean_list_items(_row.get('pain_points'), blocked=NO_PAIN_TOKENS))
        _all_pos.extend(_clean_list_items(_row.get('positive_aspects')))

    pain_counts = Counter(_all_pain)
    positive_counts = Counter(_all_pos)
    print(f"✓ Rebuilt counts | pain terms: {len(pain_counts):,}, positive terms: {len(positive_counts):,}")

top_pain_items = [item for item, _ in pain_counts.most_common(50)]
top_positive_items = [item for item, _ in positive_counts.most_common(50)]

unified_prompt = f"""You are a senior product analyst for an Electronics company.

Below are the 50 most common PAIN POINTS and 50 most common POSITIVE FEATURES extracted from Amazon customer reviews.

PAIN POINTS:
{json.dumps(top_pain_items, indent=2)}

POSITIVE FEATURES:
{json.dumps(top_positive_items, indent=2)}

Create a UNIFIED set of 10-14 product dimension categories that cover BOTH pain points and positives. 
Each category represents an area of product performance (e.g., "Sound & Audio Quality" would include both the pain point "poor sound" and the positive "great sound").

Any items that truly don't fit any category should go into "Others".

Return JSON:
{{
  "categories": {{
    "Category Name": {{
      "description": "1-sentence description of what this dimension covers",
      "pain_points": ["matching pain points from the list above"],
      "positives": ["matching positives from the list above"]
    }},
    "Others": {{
      "description": "Items that don't fit any specific product dimension",
      "pain_points": ["..."],
      "positives": ["..."]
    }}
  }}
}}

RULES:
- Every item from BOTH lists must appear in exactly one category
- Category names should be clear product dimensions (e.g., "Battery & Power", "Build Quality & Durability")
- "Others" MUST exist as a catch-all for miscellaneous items
- A category CAN have only pain points or only positives — that's fine and insightful
"""

cat_response = send_message(
    [{"role": "user", "content": unified_prompt}],
    max_tokens=3000, temperature=0.1,
    response_format={"type": "json_object"}
)

pain_cat_map = {}      # pain_point → category
positive_cat_map = {}  # positive → category
category_info = {}     # category → description

if cat_response:
    try:
        cat_data = json.loads(cat_response)
        categories = cat_data.get("categories", {})
        for cat_name, detail in categories.items():
            category_info[cat_name] = detail.get("description", "")
            for pp in detail.get("pain_points", []):
                pain_cat_map[pp.lower().strip()] = cat_name
            for pos in detail.get("positives", []):
                positive_cat_map[pos.lower().strip()] = cat_name

        print(f"✓ Created {len(category_info)} unified categories\n")
        print(f"{'Category':40s} {'Pain Pts':>10s} {'Positives':>10s}")
        print("─" * 62)
        for cat_name in category_info:
            n_pain = sum(1 for v in pain_cat_map.values() if v == cat_name)
            n_pos  = sum(1 for v in positive_cat_map.values() if v == cat_name)
            print(f"  {cat_name:38s} {n_pain:8d} {n_pos:10d}")
            print(f"    ↳ {category_info[cat_name]}")
    except json.JSONDecodeError:
        print("Warning: Could not parse LLM response")

# ═══════════════════════════════════════════════════════════════════
# STEP 2: Classify remaining phrases (long tail)
#
# Notebook intent: map phrases into the unified categories; anything
# unmapped is treated as "Others" in Step 3 (explicit catch‑all).
#
# Speed knobs (still aligned with that intent):
#   CLASSIFY_LONG_TAIL — False skips Step 2 entirely (unmapped → Others).
#   MAX_UNIQUE_PHRASES_TO_CLASSIFY — only the most frequent remaining
#     distinct phrases are sent to the LLM; rarer ones → Others in Step 3.
#   CLASSIFY_BATCH_SIZE — more phrases per request = fewer API calls.
#   CLASSIFY_PARALLEL — concurrent requests (async), like Part 4.2.
# ═══════════════════════════════════════════════════════════════════

CLASSIFY_LONG_TAIL = True
MAX_UNIQUE_PHRASES_TO_CLASSIFY = 500  # None = every distinct phrase (slowest)
CLASSIFY_BATCH_SIZE = 120
CLASSIFY_PARALLEL = min(8, max(4, CONCURRENT_REQUESTS))

cat_names_list = list(category_info.keys())
valid_cats = set(category_info.keys()) | {"Others"}


def _normalize_category_label(name):
    if not isinstance(name, str):
        return None
    name = name.strip()
    if name in valid_cats:
        return name
    if name.lower() == "others":
        return "Others"
    for k in category_info:
        if k.lower() == name.lower():
            return k
    return None


async def classify_remaining_async(all_items_counter, existing_map, label):
    remaining = [
        item
        for item, _ in all_items_counter.most_common()
        if item.lower().strip() not in existing_map
    ]
    total_remaining = len(remaining)
    if not remaining:
        return existing_map

    cap = MAX_UNIQUE_PHRASES_TO_CLASSIFY
    if cap is not None and len(remaining) > cap:
        n_skip = len(remaining) - cap
        remaining = remaining[:cap]
        print(
            f"\nClassifying top {len(remaining)} of {total_remaining} remaining {label} "
            f"(by frequency); {n_skip} rarer distinct phrases → 'Others' in Step 3."
        )
    else:
        print(f"\nClassifying {len(remaining)} remaining {label}...")

    batches = [
        remaining[i : i + CLASSIFY_BATCH_SIZE]
        for i in range(0, len(remaining), CLASSIFY_BATCH_SIZE)
    ]

    async def one_batch(session, batch):
        cls_prompt = f"""Classify each {label.rstrip('s')} into ONE of these categories:
{json.dumps(cat_names_list)}

Items to classify:
{json.dumps(batch)}

Return JSON: {{"assignments": {{"item": "Category Name", ...}}}}
Use "Others" for items that truly don't fit anywhere else."""
        return await send_message_async(
            session,
            [{"role": "user", "content": cls_prompt}],
            max_tokens=2800,
            temperature=0.1,
            response_format={"type": "json_object"},
        )

    async with aiohttp.ClientSession() as session:
        for w0 in range(0, len(batches), CLASSIFY_PARALLEL):
            wave = batches[w0 : w0 + CLASSIFY_PARALLEL]
            responses = await asyncio.gather(*[one_batch(session, b) for b in wave])
            for resp in responses:
                if not resp:
                    continue
                try:
                    for item, cat in json.loads(resp).get("assignments", {}).items():
                        cat_n = _normalize_category_label(cat)
                        if cat_n:
                            existing_map[item.lower().strip()] = cat_n
                except json.JSONDecodeError:
                    pass

    print(f"  ✓ Total {label} keys in map: {len(existing_map)}")
    return existing_map


if CLASSIFY_LONG_TAIL and cat_names_list:
    pain_cat_map = await classify_remaining_async(pain_counts, pain_cat_map, "pain points")
    positive_cat_map = await classify_remaining_async(positive_counts, positive_cat_map, "positives")
else:
    if not cat_names_list:
        print("\nStep 2 skipped: no categories from Step 1 (check API / JSON).")
    else:
        print("\nStep 2 skipped (CLASSIFY_LONG_TAIL=False). Unmapped phrases → 'Others' in Step 3.")

# ═══════════════════════════════════════════════════════════════════
# STEP 3: Aggregate counts at category level
# ═══════════════════════════════════════════════════════════════════

pain_category_counts = Counter()
for pp, count in pain_counts.items():
    cat = pain_cat_map.get(pp.lower().strip(), "Others")
    pain_category_counts[cat] += count

positive_category_counts = Counter()
for pos, count in positive_counts.items():
    cat = positive_cat_map.get(pos.lower().strip(), "Others")
    positive_category_counts[cat] += count

all_category_names = sorted(set(
    list(pain_category_counts.keys()) + list(positive_category_counts.keys())
))

print(f"\n{'═' * 70}")
print("  UNIFIED CATEGORY SCORECARD")
print(f"{'═' * 70}")
print(f"\n  {'Category':35s} {'Pain Pts':>10s} {'Positives':>10s} {'Net':>8s}")
print("  " + "─" * 65)
for cat in all_category_names:
    p = pain_category_counts.get(cat, 0)
    pos = positive_category_counts.get(cat, 0)
    net = pos - p
    indicator = "🟢" if net > 0 else "🔴" if net < 0 else "🟡"
    print(f"  {indicator} {cat:33s} {p:10,d} {pos:10,d} {net:+8,d}")

✓ Created 14 unified categories

Category                                   Pain Pts  Positives
──────────────────────────────────────────────────────────────
  Sound & Audio Quality                        13          8
    ↳ Covers the fidelity, clarity, volume, and bass performance of the audio output.
  Reliability & Lifespan                        7          7
    ↳ Concerns related to the product failing, breaking, or its overall longevity.
  Battery & Power                               2          2
    ↳ Relates to the performance, duration, and reliability of the internal battery.
  Build Quality & Materials                     3          4
    ↳ Focuses on the physical construction, materials used, and perceived sturdiness of the device.
  Comfort & Fit (Ergonomics)                    5          4
    ↳ Concerns the physical interaction between the user and the product, especially regarding fit and comfort.
  Connectivity & Signal                         4          0
    ↳ Per

In [25]:
# ═══════════════════════════════════════════════════════════════════
# VIZ 1: BUTTERFLY CHART — Pain vs Positive side by side per category
#   Left (red) = pain points.  Right (green) = positives.
#   This is the single most useful chart in the entire project.
# ═══════════════════════════════════════════════════════════════════

required_vars = ['all_category_names', 'pain_category_counts', 'positive_category_counts']
missing = [v for v in required_vars if v not in globals()]

if missing:
    print("Run prerequisite cells first: 26 -> 28 -> 34, then run this cell.")
    print(f"Missing variables: {', '.join(missing)}")
else:
    cats_for_chart = [c for c in all_category_names if c != "Others"]
    cats_for_chart = sorted(cats_for_chart,
        key=lambda c: pain_category_counts.get(c, 0) + positive_category_counts.get(c, 0),
        reverse=False)

    fig_butterfly = go.Figure()
    pain_vals = [pain_category_counts.get(c, 0) for c in cats_for_chart]
    pos_vals = [positive_category_counts.get(c, 0) for c in cats_for_chart]

    fig_butterfly.add_trace(go.Bar(
        name='Pain Points',
        y=cats_for_chart,
        x=[-v for v in pain_vals],
        orientation='h',
        marker_color='#e74c3c',
        text=pain_vals,
        texttemplate='%{text:,}',
        textposition='inside',
        insidetextanchor='middle',
        hovertemplate='%{y}<br>Pain Points: %{text:,}<extra></extra>'
    ))
    fig_butterfly.add_trace(go.Bar(
        name='Positive Features',
        y=cats_for_chart,
        x=pos_vals,
        orientation='h',
        marker_color='#27ae60',
        text=pos_vals,
        texttemplate='%{text:,}',
        textposition='inside',
        insidetextanchor='middle',
        hovertemplate='%{y}<br>Positive Features: %{text:,}<extra></extra>'
    ))
    fig_butterfly.update_layout(
        barmode='relative',
        title='Category Scorecard: Pain Points (←Red) vs Positives (Green→)',
        xaxis_title='← Pain Points | Positive Features →',
        height=620,
        width=1300,
        bargap=0.22,
        legend=dict(orientation='h', x=0.02, y=1.12),
        margin=dict(l=230, r=60, t=120, b=60)
    )
    fig_butterfly.show()

# ═══════════════════════════════════════════════════════════════════
# VIZ 2: TREEMAPS — one for pain points, one for positives
#   Drill into each category to see specific items.
# ═══════════════════════════════════════════════════════════════════

def build_treemap_data(counter, cat_map, label):
    data = []
    for item, count in counter.most_common(80):
        cat = cat_map.get(item.lower().strip(), "Others")
        data.append({"category": cat, "item": item, "count": count, "type": label})
    return pd.DataFrame(data)

pain_tree_df = build_treemap_data(pain_counts, pain_cat_map, "Pain Point")
pos_tree_df  = build_treemap_data(positive_counts, positive_cat_map, "Positive")

fig_tree_pain = px.treemap(
    pain_tree_df, path=['category', 'item'], values='count',
    title='Pain Point Categories — Treemap (bigger = more frequent)',
    color='count', color_continuous_scale='Reds'
)
fig_tree_pain.update_layout(height=550, width=950)
fig_tree_pain.show()

fig_tree_pos = px.treemap(
    pos_tree_df, path=['category', 'item'], values='count',
    title='Positive Feature Categories — Treemap (bigger = more frequent)',
    color='count', color_continuous_scale='Greens'
)
fig_tree_pos.update_layout(height=550, width=950)
fig_tree_pos.show()

# ═══════════════════════════════════════════════════════════════════
# VIZ 3: LOW vs HIGH RATING — which categories DRIVE bad ratings?
# ═══════════════════════════════════════════════════════════════════

def segment_category_counts(segment_df, column, cat_map):
    counts = Counter()
    for _, row in segment_df.iterrows():
        if isinstance(row.get(column), list):
            for item in row[column]:
                counts[cat_map.get(item.lower().strip(), "Others")] += 1
    return counts

low_pain_cat  = segment_category_counts(low_rated,  'pain_points', pain_cat_map)
high_pain_cat = segment_category_counts(high_rated, 'pain_points', pain_cat_map)
low_pos_cat   = segment_category_counts(low_rated,  'positive_aspects', positive_cat_map)
high_pos_cat  = segment_category_counts(high_rated, 'positive_aspects', positive_cat_map)

# Build aligned vectors once (cleaner than adding one trace per category)
pain_low_vals = [low_pain_cat.get(cat, 0) for cat in cats_for_chart]
pain_high_vals = [high_pain_cat.get(cat, 0) for cat in cats_for_chart]
pos_low_vals = [low_pos_cat.get(cat, 0) for cat in cats_for_chart]
pos_high_vals = [high_pos_cat.get(cat, 0) for cat in cats_for_chart]

fig_seg = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=['Pain Points by Rating Segment', 'Positives by Rating Segment'],
    horizontal_spacing=0.22
)

# Pain subplot
fig_seg.add_trace(
    go.Bar(
        name='Low-rated (1-2★)',
        y=cats_for_chart,
        x=pain_low_vals,
        orientation='h',
        marker_color='#c0392b',
        text=pain_low_vals,
        textposition='outside',
        cliponaxis=False
    ),
    row=1,
    col=1
)
fig_seg.add_trace(
    go.Bar(
        name='High-rated (4-5★)',
        y=cats_for_chart,
        x=pain_high_vals,
        orientation='h',
        marker_color='#f5b7b1',
        text=pain_high_vals,
        textposition='outside',
        cliponaxis=False
    ),
    row=1,
    col=1
)

# Positive subplot
fig_seg.add_trace(
    go.Bar(
        name='Low-rated (1-2★)',
        y=cats_for_chart,
        x=pos_low_vals,
        orientation='h',
        marker_color='#76d7c4',
        text=pos_low_vals,
        textposition='outside',
        cliponaxis=False,
        showlegend=False
    ),
    row=1,
    col=2
)
fig_seg.add_trace(
    go.Bar(
        name='High-rated (4-5★)',
        y=cats_for_chart,
        x=pos_high_vals,
        orientation='h',
        marker_color='#1e8449',
        text=pos_high_vals,
        textposition='outside',
        cliponaxis=False,
        showlegend=False
    ),
    row=1,
    col=2
)

fig_seg.update_layout(
    height=720,
    width=1500,
    barmode='group',
    bargap=0.22,
    title_text='Category Breakdown: Low-Rated vs High-Rated',
    legend=dict(orientation='h', y=1.08, x=0.01),
    margin=dict(l=220, r=80, t=110, b=60)
)
fig_seg.update_xaxes(title_text='Mention Count', tickformat=',d', row=1, col=1)
fig_seg.update_xaxes(title_text='Mention Count', tickformat=',d', row=1, col=2)
fig_seg.update_yaxes(autorange='reversed', row=1, col=1)
fig_seg.update_yaxes(autorange='reversed', row=1, col=2)
fig_seg.show()

# ═══════════════════════════════════════════════════════════════════
# STRATEGIC INSIGHTS — automated interpretation
# ═══════════════════════════════════════════════════════════════════

print(f"\n{'═' * 70}")
print("  STRATEGIC CATEGORY INSIGHTS")
print(f"{'═' * 70}")

print("\n🔴 PROBLEM CATEGORIES (pain > positive — fix these first):")
for cat in all_category_names:
    p = pain_category_counts.get(cat, 0)
    pos = positive_category_counts.get(cat, 0)
    if p > pos and cat != "Others":
        print(f"   {cat:35s}  Pain: {p:,}  Positive: {pos:,}  (gap: {p - pos:,})")

print("\n🟢 STRENGTH CATEGORIES (positive > pain — double down here):")
for cat in all_category_names:
    p = pain_category_counts.get(cat, 0)
    pos = positive_category_counts.get(cat, 0)
    if pos > p and cat != "Others":
        print(f"   {cat:35s}  Positive: {pos:,}  Pain: {p:,}  (surplus: {pos - p:,})")

print("\n⚡ POLARIZING CATEGORIES (high in BOTH — inconsistent across products):")
for cat in all_category_names:
    p = pain_category_counts.get(cat, 0)
    pos = positive_category_counts.get(cat, 0)
    if p > 50 and pos > 50 and cat != "Others":
        ratio = min(p, pos) / max(p, pos)
        if ratio > 0.3:
            print(f"   {cat:35s}  Pain: {p:,}  Positive: {pos:,}  → BIG OPPORTUNITY to differentiate")


══════════════════════════════════════════════════════════════════════
  STRATEGIC CATEGORY INSIGHTS
══════════════════════════════════════════════════════════════════════

🔴 PROBLEM CATEGORIES (pain > positive — fix these first):
   Compatibility                        Pain: 184  Positive: 22  (gap: 162)
   Connectivity & Signal                Pain: 744  Positive: 733  (gap: 11)

🟢 STRENGTH CATEGORIES (positive > pain — double down here):
   Battery & Power                      Positive: 720  Pain: 510  (surplus: 210)
   Build Quality & Materials            Positive: 2,283  Pain: 587  (surplus: 1,696)
   Comfort & Fit (Ergonomics)           Positive: 2,092  Pain: 1,049  (surplus: 1,043)
   Performance & Functionality          Positive: 2,941  Pain: 863  (surplus: 2,078)
   Physical Attributes & Design         Positive: 2,686  Pain: 605  (surplus: 2,081)
   Product Description & Accuracy       Positive: 455  Pain: 234  (surplus: 221)
   Reliability & Lifespan               Positive: 3

---

## **Part 6: Optional Advanced Analysis - Embedding-Based Clustering**

### **TODO 6.1**: Generate embeddings for reviews

**Your task** (if time permits): 
1. Create embeddings for review texts
2. Reduce dimensions with PCA + t-SNE
3. Cluster reviews using K-Means
4. Visualize clusters

**Why clustering?**
- Discover natural groupings of similar reviews
- Identify distinct customer segments or issues
- Find patterns not captured by keywords alone

**Hints**:
- Use the `embedder` already initialized
- Process in batches to avoid memory issues


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 
# ═══════════════════════════════════════════════════════════════════
#
# WHAT: Turn each review into a numeric vector (embedding), then
#       cluster those vectors to find natural groupings the keyword-
#       based analysis in Part 5 might miss.
#
# PIPELINE:
#   review text → embedding (1536-dim via API) → PCA (50-dim)
#   → K-Means clustering → t-SNE (2-dim for plotting)
# ─────────────────────────────────────────────────────────────────

EMBEDDINGS_FILE = "electronics_embeddings.npy"

# Step 1: Prepare texts
review_texts = insights_df['review_body'].fillna('').astype(str).tolist()
review_texts = [text[:1000] for text in review_texts]
print(f"Prepared {len(review_texts):,} texts for embedding")

# Step 2: Generate or load embeddings
if os.path.exists(EMBEDDINGS_FILE):
    embeddings = np.load(EMBEDDINGS_FILE)
    print(f"✓ Loaded cached embeddings from {EMBEDDINGS_FILE}  |  shape: {embeddings.shape}")
else:
    print("Generating embeddings via API (this may take a few minutes)...")
    embeddings_list = embed_texts_api(review_texts, model=EMBEDDING_MODEL)
    embeddings = np.array(embeddings_list)
    np.save(EMBEDDINGS_FILE, embeddings)
    print(f"✓ Generated & saved embeddings  |  shape: {embeddings.shape}")

# Step 3: PCA — reduce from 1536 dims → 50 dims
# WHY? Makes clustering faster and removes noise.  50 components
#      typically capture 60-70% of variance, which is plenty.
print("\nApplying PCA (1536 → 50 dimensions)...")
pca = PCA(n_components=50, random_state=42)
embeddings_pca = pca.fit_transform(embeddings)
print(f"  Variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Step 4: Find optimal k using elbow + silhouette
print("\nSearching for optimal cluster count...")
k_range = range(3, 16)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings_pca)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(embeddings_pca, labels))

fig_k = make_subplots(rows=1, cols=2, subplot_titles=('Elbow Method (Inertia)', 'Silhouette Score'))
fig_k.add_trace(go.Scatter(x=list(k_range), y=inertias, mode='lines+markers'), row=1, col=1)
fig_k.add_trace(go.Scatter(x=list(k_range), y=sil_scores, mode='lines+markers'), row=1, col=2)
fig_k.update_layout(height=350, width=900, showlegend=False,
                     title_text='Choosing Number of Clusters (k)')
fig_k.update_xaxes(title_text='k', row=1, col=1)
fig_k.update_xaxes(title_text='k', row=1, col=2)
fig_k.show()

best_k = list(k_range)[np.argmax(sil_scores)]
print(f"\n✓ Best k by silhouette: {best_k}  (score: {max(sil_scores):.3f})")

# Step 5: Final K-Means with chosen k
N_CLUSTERS = best_k
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
insights_df['cluster'] = kmeans.fit_predict(embeddings_pca)

print(f"\nCluster sizes (k={N_CLUSTERS}):")
for cid, cnt in insights_df['cluster'].value_counts().sort_index().items():
    print(f"  Cluster {cid}: {cnt:,} reviews ({cnt/len(insights_df)*100:.1f}%)")

# Step 6: t-SNE for 2D visualization
# WHY t-SNE? It preserves local neighborhood structure so similar
#   reviews appear near each other on the plot.
print("\nApplying t-SNE (50 → 2 dimensions)... this may take a minute.")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
emb_2d = tsne.fit_transform(embeddings_pca)
insights_df['tsne_x'] = emb_2d[:, 0]
insights_df['tsne_y'] = emb_2d[:, 1]

# Step 7: Interactive scatter plot
fig_clusters = px.scatter(
    insights_df,
    x='tsne_x', y='tsne_y',
    color='cluster',
    hover_data=['star_rating', 'sentiment', 'product_title'],
    title=f'Review Clusters (K-Means k={N_CLUSTERS})',
    labels={'cluster': 'Cluster ID'},
    color_continuous_scale='Viridis'
)
fig_clusters.update_layout(height=600, width=900)
fig_clusters.show()

# Step 8: Use LLM to label each cluster
print("\nLabeling clusters with LLM...")
cluster_analyses = {}
for cid in range(N_CLUSTERS):
    cluster_df = insights_df[insights_df['cluster'] == cid]
    sample = cluster_df.sample(min(15, len(cluster_df)), random_state=42)
    reviews_text = "\n".join([
        f"Review (Rating {r['star_rating']}/5): {str(r['review_body'])[:400]}"
        for _, r in sample.iterrows()
    ])
    prompt = f"""Analyze these {len(cluster_df)} reviews from Cluster {cid}.

{reviews_text}

Return JSON:
{{
  "cluster_theme": "main theme in 5-10 words",
  "common_topics": ["topic1", "topic2", "topic3"],
  "typical_sentiment": "positive/negative/mixed",
  "key_characteristics": "what makes this cluster distinct (1-2 sentences)"
}}"""
    resp = send_message(
        [{"role": "user", "content": prompt}],
        max_tokens=500, temperature=0.1,
        response_format={"type": "json_object"}
    )
    if resp:
        try:
            analysis = json.loads(resp)
            analysis['size'] = len(cluster_df)
            analysis['avg_rating'] = cluster_df['star_rating'].mean()
            cluster_analyses[cid] = analysis
        except json.JSONDecodeError:
            pass

print(f"\n{'═' * 70}")
print("CLUSTER ANALYSIS RESULTS")
print(f"{'═' * 70}")
for cid, a in cluster_analyses.items():
    print(f"\nCluster {cid}: {a.get('cluster_theme', 'N/A')}")
    print(f"  Size: {a['size']:,}  |  Avg Rating: {a['avg_rating']:.2f}")
    print(f"  Sentiment: {a.get('typical_sentiment', 'N/A')}")
    print(f"  Topics: {', '.join(a.get('common_topics', []))}")
    print(f"  Key: {a.get('key_characteristics', 'N/A')}")

Prepared 29,942 texts for embedding
✓ Loaded cached embeddings from electronics_embeddings.npy  |  shape: (29942, 1536)

Applying PCA (1536 → 50 dimensions)...
  Variance explained: 49.7%

Searching for optimal cluster count...



✓ Best k by silhouette: 12  (score: 0.095)

Cluster sizes (k=12):
  Cluster 0: 1,758 reviews (5.9%)
  Cluster 1: 3,225 reviews (10.8%)
  Cluster 2: 1,463 reviews (4.9%)
  Cluster 3: 2,297 reviews (7.7%)
  Cluster 4: 3,846 reviews (12.8%)
  Cluster 5: 3,135 reviews (10.5%)
  Cluster 6: 1,257 reviews (4.2%)
  Cluster 7: 5,016 reviews (16.8%)
  Cluster 8: 3,044 reviews (10.2%)
  Cluster 9: 2,263 reviews (7.6%)
  Cluster 10: 1,595 reviews (5.3%)
  Cluster 11: 1,043 reviews (3.5%)

Applying t-SNE (50 → 2 dimensions)... this may take a minute.



Labeling clusters with LLM...

══════════════════════════════════════════════════════════════════════
CLUSTER ANALYSIS RESULTS
══════════════════════════════════════════════════════════════════════

Cluster 0: Power accessories, charging cables, and surge protection devices
  Size: 1,758  |  Avg Rating: 4.29
  Sentiment: mixed
  Topics: Power supply and charging functionality, Physical design and usability issues (e.g., size, handles, spacing), Connectivity and compatibility problems
  Key: This cluster heavily features reviews for various power-related accessories like adapters, surge protectors, and power supplies, showing a mix of satisfaction with core function and complaints about physical design flaws or connectivity issues.

Cluster 1: Widespread product failure and poor reliability issues
  Size: 3,225  |  Avg Rating: 1.60
  Sentiment: negative
  Topics: Product stopped working/died prematurely, Poor build quality and component failure (switches, power), Inaccurate product cla

---

## **Part 7: Temporal Trend Analysis**

### **TODO 7.1**: Analyze how pain points evolved over time

**Your task**: 
1. Group pain points by year
2. Track frequency of top pain points over time
3. Create line chart showing trends
4. Analyze sentiment trends over time

**Questions to answer**:
- Are customers' concerns changing over time?
- Is overall sentiment improving or declining?
- Are there seasonal patterns?

**Hints**:
- Use `insights_df.groupby('year')`
- Track top 5-10 pain points across years
- Use plotly line charts for interactivity
- Reference: M9.1 Example (Part 7)

In [27]:
# WHY TEMPORAL? A static "top pain points" list doesn't tell you if
# "poor battery" is getting worse or better over time.  A company
# entering the market in 2015 cares about *current* problems, not
# ones that were fixed in 2010.

# ── 1. Pain points by year ──────────────────────────────────────────
pain_by_year = {}
sentiment_by_year = {}

for _, row in insights_df.iterrows():
    year = row.get('year')
    if pd.isna(year):
        continue
    year = int(year)

    if isinstance(row.get('pain_points'), list):
        pain_by_year.setdefault(year, []).extend(row['pain_points'])

    if isinstance(row.get('sentiment'), str):
        sentiment_by_year.setdefault(year, []).append(row['sentiment'])

# Get overall top 8 pain points to track across years
top_pains_overall = [item for item, _ in pain_counts.most_common(8)]

years_sorted = sorted(pain_by_year.keys())
pain_trends = {}
for pain in top_pains_overall:
    pain_trends[pain] = [
        Counter(pain_by_year.get(y, [])).get(pain, 0)
        for y in years_sorted
    ]

# Line chart of top pain points over time
fig_trends = go.Figure()
for pain, counts_list in pain_trends.items():
    fig_trends.add_trace(go.Scatter(
        x=years_sorted, y=counts_list,
        mode='lines+markers', name=pain
    ))
fig_trends.update_layout(
    title='Top Pain Points Over Time',
    xaxis_title='Year', yaxis_title='Mention Count',
    height=450, width=900
)
fig_trends.show()

# ── 2. Sentiment trend over time ───────────────────────────────────
sentiment_trend_data = []
for year in years_sorted:
    year_sentiments = sentiment_by_year.get(year, [])
    total = len(year_sentiments)
    if total == 0:
        continue
    for sent_val in ['positive', 'negative', 'mixed', 'neutral']:
        cnt = year_sentiments.count(sent_val)
        sentiment_trend_data.append({
            'year': year,
            'sentiment': sent_val,
            'percentage': cnt / total * 100
        })

sent_trend_df = pd.DataFrame(sentiment_trend_data)
fig_sent = px.line(
    sent_trend_df, x='year', y='percentage', color='sentiment',
    title='Sentiment Distribution Over Time',
    labels={'percentage': '% of Reviews', 'year': 'Year'},
    markers=True
)
fig_sent.update_layout(height=400, width=900)
fig_sent.show()

# Print key observations
print("\nKey Temporal Observations:")
for pain in top_pains_overall[:5]:
    first_val = pain_trends[pain][0] if pain_trends[pain][0] > 0 else 1
    last_val = pain_trends[pain][-1]
    direction = "↑ increasing" if last_val > first_val else "↓ decreasing"
    print(f"  • '{pain}': {direction} ({first_val} → {last_val})")


Key Temporal Observations:
  • 'stopped working': ↑ increasing (21 → 60)
  • 'short lifespan': ↑ increasing (35 → 61)
  • 'high price': ↓ decreasing (24 → 9)
  • 'low volume': ↑ increasing (15 → 21)
  • 'did not work': ↑ increasing (12 → 26)


---

## **Part 8: Compare Top vs Bottom Products**

### **TODO 8.1**: Comparative analysis using LLM

**Your task**: 
1. Get reviews from top 5 products (from Part 3)
2. Get reviews from bottom 5 products
3. Use LLM to identify key differences
4. Generate strategic recommendations

**Hints**:
- Sample ~50 reviews from each group
- Create a comparison prompt for the LLM
- Request structured JSON output with success/failure factors
-

In [ ]:
# WHY COMPARE? The LLM can read 50 reviews from winners and 50 from
# losers, then articulate what separates them — something a bar chart
# can't do.  This gives you "the recipe for a successful product."

# Gather reviews from top/bottom products identified in Part 3
top_reviews = df[df['product_title'].isin(top_product_titles)]
bottom_reviews = df[df['product_title'].isin(bottom_product_titles)]

# Sample ~50 from each to keep prompt manageable
top_sample = top_reviews.sample(min(50, len(top_reviews)), random_state=42)
bottom_sample = bottom_reviews.sample(min(50, len(bottom_reviews)), random_state=42)

print(f"Top-rated product reviews sampled: {len(top_sample)}")
print(f"Bottom-rated product reviews sampled: {len(bottom_sample)}")

top_text = "\n".join([
    f"[{r['star_rating']}★] {str(r['review_body'])[:300]}"
    for _, r in top_sample.iterrows()
])
bottom_text = "\n".join([
    f"[{r['star_rating']}★] {str(r['review_body'])[:300]}"
    for _, r in bottom_sample.iterrows()
])

comparison_prompt = f"""You are a market analyst comparing top-rated vs bottom-rated Electronics products on Amazon.

=== TOP-RATED PRODUCT REVIEWS ===
{top_text[:6000]}

=== BOTTOM-RATED PRODUCT REVIEWS ===
{bottom_text[:6000]}

Analyze the differences and return JSON:
{{
  "success_factors": ["factor 1", "factor 2", "..."],
  "failure_factors": ["factor 1", "factor 2", "..."],
  "quality_differentiators": "what top products do differently (2-3 sentences)",
  "common_complaints_bottom": "what bottom products consistently get wrong (2-3 sentences)",
  "strategic_recommendations": ["recommendation 1", "recommendation 2", "..."],
  "market_positioning_advice": "how a new entrant should position themselves (2-3 sentences)"
}}"""

comparison_response = send_message(
    [{"role": "user", "content": comparison_prompt}],
    max_tokens=1500, temperature=0.1,
    response_format={"type": "json_object"}
)

comparison = None
if comparison_response:
    try:
        comparison = json.loads(comparison_response)
    except json.JSONDecodeError:
        print("Warning: Could not parse LLM response as JSON")

if comparison:
    print(f"\n{'═' * 70}")
    print("TOP vs BOTTOM PRODUCT COMPARISON")
    print(f"{'═' * 70}")

    print("\n✅ Success Factors (what winners do right):")
    for f in comparison.get('success_factors', []):
        print(f"   • {f}")

    print("\n❌ Failure Factors (what losers get wrong):")
    for f in comparison.get('failure_factors', []):
        print(f"   • {f}")

    print(f"\n📊 Quality Differentiators:\n   {comparison.get('quality_differentiators', 'N/A')}")
    print(f"\n📉 Common Complaints (Bottom):\n   {comparison.get('common_complaints_bottom', 'N/A')}")

    print("\n🎯 Strategic Recommendations:")
    for r in comparison.get('strategic_recommendations', []):
        print(f"   • {r}")

    print(f"\n🏁 Market Positioning Advice:\n   {comparison.get('market_positioning_advice', 'N/A')}")
else:
    print("Comparison failed — check API key / model.")

Top-rated product reviews sampled: 50
Bottom-rated product reviews sampled: 50

══════════════════════════════════════════════════════════════════════
TOP vs BOTTOM PRODUCT COMPARISON
══════════════════════════════════════════════════════════════════════

✅ Success Factors (what winners do right):
   • Functionality: Works exactly as expected, perfectly, or as advertised.
   • Quality Perception: High build quality, feeling substantial (weight), and sturdiness.
   • Performance Improvement: Noticeable positive impact on speed, audio quality, or system performance (e.g., memory upgrades).
   • Ease of Use: Easy to install or use right out of the box.
   • Value: Great price or bargain for the quality received.

❌ Failure Factors (what losers get wrong):
   • Fundamental incompatibility (e.g., analog VGA to digital HDMI conversion without a converter).
   • Complete non-functionality ('does not work one little bit').
   • Poor component quality leading to immediate failure (e.g., static 

---

## **Part 9: Executive Summary**

### **TODO 9.1**: Generate executive summary using LLM

**Your task**: 
1. Compile all key findings
2. Use LLM to generate business-focused executive summary
3. Include specific recommendations for market entry

**Summary should include**:
- Market opportunity assessment
- Top 3-5 key findings
- Critical success factors
- Areas to avoid
- Strategic recommendations
- Next steps for research

**Hints**:
- Provide LLM with aggregated data (top pain points, positives, etc.)
- Request structured JSON output
- Focus on actionable insights


In [29]:
# WHY AN EXEC SUMMARY? Everything above is raw analysis.  The exec
# summary synthesizes it into a 1-page business brief a VP can read
# in 5 minutes and make a go/no-go decision.

summary_data = {
    "category": "Electronics",
    "total_reviews_analyzed": len(insights_df),
    "date_range": f"{df['review_date'].min().date()} to {df['review_date'].max().date()}",
    "overall_avg_rating": round(df['star_rating'].mean(), 2),
    "sentiment_breakdown": dict(sentiment_counts),
    "top_pain_points": [f"{item} ({count})" for item, count in pain_counts.most_common(15)],
    "top_positives": [f"{item} ({count})" for item, count in positive_counts.most_common(15)],
    "top_themes": [f"{item} ({count})" for item, count in theme_counts.most_common(10)],
    "comparison_highlights": comparison if comparison else "N/A"
}

exec_prompt = f"""You are a senior market analyst. Based on the following analysis of {summary_data['total_reviews_analyzed']:,} Amazon Electronics reviews, create an executive summary for a company considering market entry.

DATA:
{json.dumps(summary_data, indent=2)}

Return JSON with these sections:
{{
  "market_opportunity": "2-3 sentence assessment of the opportunity",
  "key_findings": ["finding 1", "finding 2", "finding 3", "finding 4", "finding 5"],
  "critical_success_factors": ["factor 1", "factor 2", "factor 3"],
  "areas_to_avoid": ["area 1", "area 2", "area 3"],
  "strategic_recommendations": ["rec 1", "rec 2", "rec 3", "rec 4"],
  "next_steps": ["step 1", "step 2", "step 3"]
}}"""

exec_response = send_message(
    [{"role": "user", "content": exec_prompt}],
    max_tokens=2000, temperature=0.1,
    response_format={"type": "json_object"}
)

executive_summary = None
if exec_response:
    try:
        executive_summary = json.loads(exec_response)
        print("✓ Executive summary generated successfully!")
    except json.JSONDecodeError:
        print("Warning: Could not parse executive summary as JSON")
else:
    print("Failed to generate executive summary — check API.")

✓ Executive summary generated successfully!


### **TODO 9.2**: Display and save executive summary

**Your task**: Format and display the executive summary clearly.

**Hints**:
- Print each section with clear headers
- Use numbered lists for findings and recommendations
- Save to a separate text file or markdown file for reporting

In [32]:
if executive_summary:
    print("=" * 80)
    print("   EXECUTIVE SUMMARY: Electronics Market Entry Analysis")
    print("=" * 80)

    print(f"\n📈 MARKET OPPORTUNITY")
    print(f"   {executive_summary.get('market_opportunity', 'N/A')}")

    print(f"\n🔍 KEY FINDINGS")
    for i, finding in enumerate(executive_summary.get('key_findings', []), 1):
        print(f"   {i}. {finding}")

    print(f"\n✅ CRITICAL SUCCESS FACTORS")
    for f in executive_summary.get('critical_success_factors', []):
        print(f"   • {f}")

    print(f"\n⚠️  AREAS TO AVOID")
    for a in executive_summary.get('areas_to_avoid', []):
        print(f"   • {a}")

    print(f"\n🎯 STRATEGIC RECOMMENDATIONS")
    for i, r in enumerate(executive_summary.get('strategic_recommendations', []), 1):
        print(f"   {i}. {r}")

    print(f"\n📋 NEXT STEPS")
    for s in executive_summary.get('next_steps', []):
        print(f"   → {s}")

    print("\n" + "=" * 80)

    # Save all results to a comprehensive JSON file
    comprehensive_results = {
        "executive_summary": executive_summary,
        "top_vs_bottom_comparison": comparison,
        "cluster_analyses": {str(k): v for k, v in cluster_analyses.items()} if 'cluster_analyses' in dir() else {},
        "aggregated_counts": {
            "pain_points": dict(pain_counts.most_common(30)),
            "positive_aspects": dict(positive_counts.most_common(30)),
            "themes": dict(theme_counts.most_common(20)),
            "factors": dict(factor_counts.most_common(20)),
            "sentiment": dict(sentiment_counts)
        },
        "metadata": {
            "category": "Electronics",
            "reviews_analyzed": len(insights_df),
            "date_range": summary_data['date_range']
        }
    }

    RESULTS_FILE = "electronics_comprehensive_results.json"
    with open(RESULTS_FILE, 'w') as f:
        json.dump(comprehensive_results, f, indent=2)
    print(f"\n✓ Saved comprehensive results to {RESULTS_FILE}")

    # Also save executive summary as a readable markdown file
    SUMMARY_FILE = "electronics_executive_summary.md"
    with open(SUMMARY_FILE, 'w') as f:
        f.write("# Executive Summary: Electronics Market Entry Analysis\n\n")
        f.write(f"## Market Opportunity\n{executive_summary.get('market_opportunity', 'N/A')}\n\n")
        f.write("## Key Findings\n")
        for i, finding in enumerate(executive_summary.get('key_findings', []), 1):
            f.write(f"{i}. {finding}\n")
        f.write("\n## Critical Success Factors\n")
        for csf in executive_summary.get('critical_success_factors', []):
            f.write(f"- {csf}\n")
        f.write("\n## Areas to Avoid\n")
        for a in executive_summary.get('areas_to_avoid', []):
            f.write(f"- {a}\n")
        f.write("\n## Strategic Recommendations\n")
        for i, r in enumerate(executive_summary.get('strategic_recommendations', []), 1):
            f.write(f"{i}. {r}\n")
        f.write("\n## Next Steps\n")
        for s in executive_summary.get('next_steps', []):
            f.write(f"- {s}\n")
    print(f"✓ Saved executive summary to {SUMMARY_FILE}")
else:
    print("No executive summary to display.")

   EXECUTIVE SUMMARY: Electronics Market Entry Analysis

📈 MARKET OPPORTUNITY
   The Amazon Electronics market shows strong overall customer satisfaction (70% positive/mixed sentiment) but is plagued by recurring reliability issues, presenting a clear opportunity for new entrants focusing on quality and clear communication. Success hinges on addressing core pain points related to product lifespan and fundamental functionality.

🔍 KEY FINDINGS
   1. Reliability is the primary concern, with 'stopped working' and 'short lifespan' being the top two pain points cited by customers.
   2. Sound quality is the most discussed theme (6,602 mentions), indicating high customer expectation and sensitivity in this area.
   3. Positive sentiment is strongly driven by products that 'work well' and offer 'great sound' or 'good price,' suggesting value perception is tied to core performance.
   4. A significant portion of negative reviews stems from fundamental incompatibility or complete product failur

## **Limitations and Future Work**

### **Limitations**

- LLM extraction can normalize language inconsistently across similar reviews (for example, synonymous phrases may be split into separate labels).
- The analysis relies on review text quality; short or vague reviews may hide implicit pain points.
- Model/provider behavior can vary over time, so reruns may produce small changes in category assignments.
- This is observational review data, so findings are directional signals, not causal proof.

### **Future Work**

- Validate a small sample manually (about 100-200 reviews) to confirm extraction quality.
- Merge similar phrases (for example, "stopped working" and "did not work") so issue counts are cleaner.
- Re-run this analysis monthly or quarterly to track whether key pain points are improving.
- Do a deeper review of the lowest-rated clusters/products to identify highest-impact fixes.
